# 1、流式调用
举例:

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage

#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=init_chat_model(
    # model="deepseek-v4-flash",
    # model_provider="deepseek",
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)


In [ ]:
for chunk in model.stream("帮我解释一下什么是人工智能"):
     print(chunk.text,end="",flush=True)


# 2、批量调用
举例1:一次性接收所有响应

In [ ]:
messages = [
"你好，你是谁？",
"2 + 3 * 5 = ?",
"中国首都在哪里？"
]
responses = model.batch(messages)
for response in responses:
    print(response)


举例3：按完成的顺序接收反应

In [ ]:
messages = [
"你好，你是谁？",
"2 + 3 * 5 = ?",
"中国首都在哪里？"
]
responses = model.batch_as_completed(messages)
for response in responses:
    print(response)

举例3：性能对比
使用batch

In [ ]:
# 准备多个输入
inputs = [
"翻译成英文：春天来了",
"翻译成英文：夏天很热",
"翻译成英文：秋天落叶",
"翻译成英文：冬天下雪"
]
# ✅ 批量调用（高效）
import time
start = time.time()
responses = model.batch(inputs)
batch_time = time.time() - start
print("批量调用结果：")
for i, response in enumerate(responses):
    print(f"{i+1}. {response.content}")
print(f"耗时: {batch_time:.2f}秒\n")

作为对比，演示循环调用invoke

In [ ]:
# ❌ 循环调用（低效，仅用于对比）
inputs = [
"翻译成英文：春天来了",
"翻译成英文：夏天很热",
"翻译成英文：秋天落叶",
"翻译成英文：冬天下雪"
]
start = time.time()
loop_responses = []
for inp in inputs:
    response = model.invoke(inp)
loop_responses.append(response)

loop_time = time.time() - start
for i, response in enumerate(responses):
    print(f"{i+1}. {response.content}")
print(f"循环调用耗时: {loop_time:.2f}秒")
print(f"批量调用节省: {((loop_time - batch_time) / loop_time * 100):.1f}%")